## Installs

In [1]:
RUNINSTALLS = False

if RUNINSTALLS:
  !pip install openpyxl
  !pip install --upgrade google-cloud-aiplatform


## Notebook Setup

In [2]:
from IPython.display import HTML, display
import IPython

def set_css(arg=None):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
IPython.core.getipython.get_ipython().events.register('pre_run_cell', set_css)

import logging
import sys
format_string = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
logger = logging.getLogger()
fhandler = logging.FileHandler(filename='notebook.log', mode='a')
formatter = logging.Formatter(format_string)
fhandler.setFormatter(formatter)
#logger.addHandler(fhandler)  #uncomment if you want a log file
logging.basicConfig(format=format_string,
                     level=logging.INFO, stream=sys.stdout)
logger.setLevel(logging.INFO)


## Imports

In [3]:
from io import BytesIO
import datetime
import yaml
import json
from pathlib import Path
import pandas as pd
import openpyxl
import vertexai
from google.cloud import storage
from vertexai.generative_models import GenerationConfig, GenerativeModel, Part
from vertexai.preview import caching

## Project setup

In [4]:
BUCKET_NAME = "uk-bh-experiments-argolis-us"
FOLDER_PATH = "subsea7/hseq_data/"
MODEL_NAME = "gemini-1.5-pro-001"

vertexai.init(project="uk-bh-experiments-argolis", location="us-central1")

In [5]:
def clean_nones(value):
    """
    Recursively remove all None values from dictionaries and lists, and returns
    the result as a new dictionary or list.
    """
    if isinstance(value, list):
        return [clean_nones(x) for x in value if x is not None]
    elif isinstance(value, dict):
        return {
            key: clean_nones(val)
            for key, val in value.items()
            if val is not None
        }
    else:
        return value

def get_description_column(df):
  column_names = list(df)
  for column_name in column_names:
      if "desc" in column_name.lower():
         return column_name
  #if none found, return the 4th column
  return column_names[3]


def get_excel_sheet( file_name, sheet_name=None, header=0, mandatory_columns=None):

  storage_client = storage.Client()
  bucket = storage_client.bucket(BUCKET_NAME)
  blob = bucket.blob(file_name)
  logging.debug(f"Got file {file_name}")

  with blob.open("rb") as f:
      file_bytes = BytesIO(f.read())

  logging.info(f"read file {file_name}")

  openpyxl.reader.excel.warnings.simplefilter(action='ignore')

  if sheet_name is None:
    sheet_names = pd.ExcelFile(file_bytes,  engine='openpyxl').sheet_names
    print(f"Available sheets:")
    for sheet_name  in sheet_names:
        print(f"{sheet_name}")
    return sheet_names
  else:
    print(f"Reading sheet {sheet_name}")

    with pd.ExcelFile(file_bytes,  engine='openpyxl') as xls:
      df = pd.read_excel(xls, sheet_name, header=header)

      print(f"Sheet {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      if mandatory_columns == None:
         mandatory_columns = [get_description_column(df)]
      logging.debug(f"Dropping all rows that have nothing in the columns: {mandatory_columns}")
      df.dropna(subset=mandatory_columns, inplace=True)
      #print(f"Cleaned Rows {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      logging.debug("Dropping columns that have have nothing in any rows")
      df.dropna(axis=1, how="all", inplace = True)

      print(f"Cleaned {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      print(f'Column names found are: {list(df.columns.values)}')

    return df


def df_to_json(df, json_name=""):
  sheet_json_str = df.to_json(orient='records')
  #print(f'{sheet_json_str[:1]}')
  sheet_json = json.loads(sheet_json_str)
  sheet_json = clean_nones(sheet_json)
  return sheet_json


def df_select_columns(df):
  print(df.head())
  print(f"{list(df)}")
  print(df.isnull().sum())


def excel2json_str(file_name, sheet_name=None, header=0, mandatory_columns=None):
  df = get_excel_sheet(file_name, sheet_name, header, mandatory_columns)
  df_json = {}
  df_json['file'] = Path(file_name).stem
  df_json['sheet'] = sheet_name
  df_json['content'] = df_to_json(df) 
  return df_json
  #return json.dumps(df_json)


def excel2yaml(file_name: str, sheet_name=None, header=0, mandatory_columns=None):
  df = get_excel_sheet(file_name, sheet_name, header, mandatory_columns)
  df_yaml = yaml.dump(df.to_dict(orient='records'),default_flow_style=None)
  return df_yaml


def upload_blob(bucket_name, file_contents, destination_blob_name):
  """Uploads a file to the bucket."""
  storage_client = storage.Client()
  bucket = storage_client.get_bucket(bucket_name)
  blob = bucket.blob(destination_blob_name)

  blob.upload_from_string(file_contents)

  print(f'File uploaded to {destination_blob_name}')

def save_excel_json(doc, folder_path = ""):
  #doc_json = json.load(doc)
  file_contents = json.dumps(doc['content'])
  file_name = f"{doc['file']}.{doc['sheet']}"
  upload_blob(BUCKET_NAME, file_contents, folder_path + file_name + ".json")

## Add Caching



In [6]:
def cache_document(documents):

    system_instruction = """
    You are an safety expert. You always stick to the facts in the sources provided, and never make up new facts.
    Now look at these lists of saftey observations, and answer the following questions.
    """

    contents = [Part.from_text(json.dumps(document)) for document in documents]

    cached_content = caching.CachedContent.create(
        model_name=MODEL_NAME,
        system_instruction=system_instruction,
        contents=contents,
        ttl=datetime.timedelta(minutes=60),
    )

    print(cached_content.name)
    return cached_content.name


def generate_from_cache(cache_id, question):


    cached_content = caching.CachedContent(cached_content_name=cache_id)

    model = GenerativeModel.from_cached_content(cached_content=cached_content)

    response = model.generate_content(question)

    return response.text


## Use JSON as TXT

In [7]:
def generate_using_text(documents, question):
    
    system_instruction = """
    You are an safety expert. You always stick to the facts in the sources provided, and never make up new facts.
    Now look at these lists of saftey observations, and answer the following questions.
    """

    prompt = f'''
        <task> Provide a helpful and factual answer to the question that the user has asked</task>
        <question>{question}</question>
        <output>As well as giving a summary answer to the question, provide 3-6 examples of obvervations from the documents that support your conclusion.  Provide output in a nicely formatted block of text.</output>
    '''

    model = GenerativeModel(MODEL_NAME)
    model = GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=[
        system_instruction,
    ],
)
    generation_config=GenerationConfig(
        temperature = 0.8
    )

    contents = [Part.from_text(json.dumps(document)) for document in documents]
    contents.append(prompt)
    response = model.count_tokens(contents)
    logging.debug(f"Prompt Token Count: {response.total_tokens}")
    logging.debug(f"Prompt Character Count: {response.total_billable_characters}")

    response = model.generate_content(contents,generation_config=generation_config,stream=False)

    # Response tokens count
    usage_metadata = response.usage_metadata
    logging.info(f"Response Prompt Token Count: {usage_metadata.prompt_token_count}")
    logging.info(f"Candidates Token Count: {usage_metadata.candidates_token_count}")
    logging.info(f"Total Token Count: {usage_metadata.total_token_count}")

    response_text = response.text

    return response_text


In [8]:
df1 = get_excel_sheet("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)
df1.head()


2024-07-12 18:31:55,618 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
Cleaned OBSERVATIONS has 34 rows and 35 columns
Column names found are: ['Obs. No', 'Date', 'Name of observer', 'Department', 'Description of observation', 'Immediate Action', 'Further Action Required', 'Status', 'Observation close out comment/date input', 'Eyes on path', 'Line of Fire', 'Use of tools & Equip', 'Communication', 'Housekeeping', 'Pre Job Planning', 'Walking/working surfaces', 'Eyes on task', "use of Barriers & warning's", 'Conformance to rules', 'PPE Head Protection', 'PPE Eye / Face Protection', 'PPE Respiratory ', 'PPE Protective Clothing', 'PPE Hand / Arm Protection', 'PPE Feet / Ankle Protection', 'Defective tools /Equipment', "Inadequate guard's and Barriers", 'Poor House keeping', 'Objects with potential to fall', 'Procedures /Work instruction issue', 'Equipment/Material; issue', 'Certification Issue', 'Imp

In [9]:
sheet_json_str = df1.to_json(orient='records')
observation = generate_using_text([sheet_json_str], "what is the most unsafe location")
print(f'{observation}')

I0000 00:00:1720805518.007256   14787 config.cc:230] gRPC experiments enabled: call_status_override_on_cancellation, event_engine_dns, event_engine_listener, http2_stats_fix, monitoring_experiment, pick_first_new, trace_record_callops, work_serializer_clears_time_cache, work_serializer_dispatch
I0000 00:00:1720805518.007500   14787 ev_epoll1_linux.cc:125] grpc epoll fd: 74


2024-07-12 18:32:04,262 - root - INFO - Response Prompt Token Count: 12518
2024-07-12 18:32:04,264 - root - INFO - Candidates Token Count: 157
2024-07-12 18:32:04,268 - root - INFO - Total Token Count: 12675
### Most Unsafe Location

The data suggests the most unsafe location is **the hangar**. This area exhibits recurring issues with lighting, potential falling objects, and material condition, indicating potential for improvement in maintenance and housekeeping.

**Supporting Observations:**

*   **Observation 122:** Vacuum hose in the hangar was too short, requiring a temporary extension and highlighting a potential maintenance oversight.
*   **Observation 132:** A light fixture was found hanging precariously due to a corroded bolt, indicating a need for better maintenance and highlighting the risk of falling objects.
*   **Observation 132 (Further Action):** This observation suggests a broader issue with hangar lighting and vibration dampers, implying a potential systemic maintenanc

I0000 00:00:1720805524.271008   14852 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720805524.275967   14852 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [23]:
json_docs = [
    excel2json_str("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)
]
question = "Where on our worksites is someone most likely to be hurt?  What were they doing?"
answer = generate_using_text(json_docs, question)
print(f'{answer}')

2024-07-12 18:45:44,310 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
Cleaned OBSERVATIONS has 34 rows and 35 columns
Column names found are: ['Obs. No', 'Date', 'Name of observer', 'Department', 'Description of observation', 'Immediate Action', 'Further Action Required', 'Status', 'Observation close out comment/date input', 'Eyes on path', 'Line of Fire', 'Use of tools & Equip', 'Communication', 'Housekeeping', 'Pre Job Planning', 'Walking/working surfaces', 'Eyes on task', "use of Barriers & warning's", 'Conformance to rules', 'PPE Head Protection', 'PPE Eye / Face Protection', 'PPE Respiratory ', 'PPE Protective Clothing', 'PPE Hand / Arm Protection', 'PPE Feet / Ankle Protection', 'Defective tools /Equipment', "Inadequate guard's and Barriers", 'Poor House keeping', 'Objects with potential to fall', 'Procedures /Work instruction issue', 'Equipment/Material; issue', 'Certification Issue', 'Imp

I0000 00:00:1720806346.071268   14787 ev_epoll1_linux.cc:125] grpc epoll fd: 78


2024-07-12 18:45:55,576 - root - INFO - Response Prompt Token Count: 5823
2024-07-12 18:45:55,579 - root - INFO - Candidates Token Count: 412
2024-07-12 18:45:55,582 - root - INFO - Total Token Count: 6235
## Areas of Concern for Potential Injuries on Worksites

Based on the provided safety observations, it appears the highest risk of injury on your worksites centers around **inadequate housekeeping and maintenance of equipment, particularly in areas with potential falling objects.** This creates trip hazards, falling object hazards, and potential exposure to sharp edges.

Here are some examples from the observations:

* **Observation 120:** A suggestion was made to weld a padeye to eliminate trip hazards. This highlights the presence of tripping hazards on deck.
* **Observation 124:** A rescue dummy was found wearing safety glasses, posing a drop hazard. This indicates a lack of attention to potential falling objects.
* **Observation 126:** A fluorescent light bulb was found hanging p

I0000 00:00:1720806355.587340   16007 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720806355.589564   16007 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [12]:
json_docs = [
    excel2json_str("subsea7/hseq_data/SevenArctic.xlsx", "OBS", 2),
    excel2json_str("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1),
    excel2json_str(f"{FOLDER_PATH}SeawayStrashnov.xlsx", "Obs_Int Register", 1),
    excel2json_str(f"{FOLDER_PATH}SeawayMoxie.xlsx", "Obs_Int Register", 10),
    excel2json_str(f"{FOLDER_PATH}SevenOceans.xlsx", "Safety SO record", 59),
    excel2json_str(f"{FOLDER_PATH}SevenPacific_Observations.xlsx", "CSB 2021", 1),
    excel2json_str(f"{FOLDER_PATH}SevenVega.xlsx", "SEVEN VEGA CSBs ", 0),

    ]    

2024-07-12 18:35:56,442 - root - INFO - read file subsea7/hseq_data/SevenArctic.xlsx
Reading sheet OBS
Sheet OBS has 7538 rows and 19 columns
Cleaned OBS has 7507 rows and 19 columns
Column names found are: ['Card\nID', 'OBS', 'Day', 'Month', 'Year', 'Time', 'Location', 'Name of Observer', 'Department', 'Description', 'Corrective/Immediate Action Taken', 'Suggested Further Action Required', 'Person Responsible\nfor Action', 'Action Undertaken', 'OPEN / Closed', 'Date\nClosed DD/MM/YYYY', 'Safety Improvement (Yes/No)', 'Unnamed: 17', 'Unnamed: 18']
2024-07-12 18:36:06,638 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
Cleaned OBSERVATIONS has 34 rows and 35 columns
Column names found are: ['Obs. No', 'Date', 'Name of observer', 'Department', 'Description of observation', 'Immediate Action', 'Further Action Required', 'Status', 'Observation close out comment/date input', 'Eyes on path', 'Line of Fir

In [14]:
for doc in json_docs:
    save_excel_json( doc, FOLDER_PATH)

File uploaded to subsea7/hseq_data/SevenArctic.OBS.json
File uploaded to subsea7/hseq_data/NormandSubsea.OBSERVATIONS.json
File uploaded to subsea7/hseq_data/SeawayStrashnov.Obs_Int Register.json
File uploaded to subsea7/hseq_data/SeawayMoxie.Obs_Int Register.json
File uploaded to subsea7/hseq_data/SevenOceans.Safety SO record.json
File uploaded to subsea7/hseq_data/SevenPacific_Observations.CSB 2021.json
File uploaded to subsea7/hseq_data/SevenVega.SEVEN VEGA CSBs .json


## Questions

Questions from John:
 - Where on our worksites is someone most likely to be hurt?  What were they doing?
 - What is the most likely way someone could be hurt?
 - Do we have a problem with doors?
 - If I was to tackle one issue on our worksites, what would it be?
 - Are our gallies safe?
 

Can AI be used on our Synergi, RA7 and MOC databases?
(Synergi is our HSEQ incident reporting tool, RA7 records risk assessments, MOC is our Management of Change tool)

 - When we change rigging offshore, do we normally increase or decrease the capacity?
 - Do we have a problem with dropped tools?

In [22]:
question = "Where on our worksites is someone most likely to be hurt?  What were they doing?"
answer = generate_using_text(json_docs, question)
print(f'{answer}')

I0000 00:00:1720806293.239498   14787 ev_epoll1_linux.cc:125] grpc epoll fd: 71


2024-07-12 18:45:42,194 - root - INFO - Response Prompt Token Count: 1736914
2024-07-12 18:45:42,196 - root - INFO - Candidates Token Count: 238
2024-07-12 18:45:42,197 - root - INFO - Total Token Count: 1737152
The most likely place for someone to be hurt on the worksites is around moving machinery, especially cranes.  There are many observations of people entering barriered off areas, not wearing correct PPE while working at height, and items being dropped from cranes.  Here are some examples: 

***

**Seven Arctic:** 

*   **Card ID 2913:** Personnel entering machinery spaces without informing ECR
*   **Card ID 3069:** e7 “Crash” when Reviewing-Updating Umbilical Load-out Task Plan(s)
*   **Card ID 3119:** Noticed a large spanner sitting 3 floors up, above the side of the LARS area in a side structure

**Normand Subsea**

*   **Card ID 130**: Basket SV – Excess of redundant single trigger RVD hooks found available for use.  These hooks have been superseded by twin trigger version
* 

I0000 00:00:1720806342.199466   15966 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720806342.200907   15966 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [16]:
question = "What is the most likely way someone could be hurt?"
answer = generate_using_text(json_docs, question)
print(f'{answer}')

I0000 00:00:1720806017.329190   14787 ev_epoll1_linux.cc:125] grpc epoll fd: 71


2024-07-12 18:41:03,806 - root - INFO - Response Prompt Token Count: 1736906
2024-07-12 18:41:03,807 - root - INFO - Candidates Token Count: 294
2024-07-12 18:41:03,808 - root - INFO - Total Token Count: 1737200
The most likely way for someone to be hurt on the Seven Arctic, the Normand Subsea, or the Seaway Strashnov is by a slip, trip, or fall. There are numerous reports of slippery surfaces, trip hazards, and unsecured items that could fall from height. 

Here are some examples of observations from the documents that support this conclusion: 

**Seven Arctic**
* **Card ID 33:** "Wet stairwell, slip hazard"
* **Card ID 90:** "Somebody throw some table napkins into the food waste bucket. Fire hazard."
* **Card ID 132:** "Fire Fighting equipment not in satisfactory state among a no of issues is bottle not screwed on properly to harness & air leaking from connection to demand valve due to not being screwed together properly."

**Normand Subsea**
* **Observation No 123:** "Reported warm 

I0000 00:00:1720806063.812956   15784 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720806063.814640   15784 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [17]:
question = "Do we have a problem with doors?"
observation = generate_using_text(json_docs, question)
print(f'{observation}')

I0000 00:00:1720806063.955517   14787 ev_epoll1_linux.cc:125] grpc epoll fd: 72


2024-07-12 18:42:07,238 - root - INFO - Response Prompt Token Count: 1736903
2024-07-12 18:42:07,243 - root - INFO - Candidates Token Count: 362
2024-07-12 18:42:07,248 - root - INFO - Total Token Count: 1737265
Yes, there are several safety observations regarding doors on the Seven Arctic, Seaway Strashnov, and Normand Subsea vessels. These observations include doors being left open, unsecured, or damaged, posing safety hazards such as pinch points, water ingress, and difficulty in emergency egress. 

Here are some examples:

**Seven Arctic**

* **Card ID 24:**  During a fire drill, some double fire doors closed such that the inner door was outside the outer door, compromising the seal.
* **Card ID 165:** WTD 10 left open. Crew to be reminded about keeping WTD closed at sea.
* **Card ID 360:** C-deck, port side door adjacent to cabin 509 does not close. Door leads to watertight door so when opened causes air rush from accommodation spaces.

**Seaway Strashnov**

* **Card # 10:** Balla

I0000 00:00:1720806127.261340   15848 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720806127.265134   15848 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [18]:
question = "If I was to tackle one issue on our worksites, what would it be?"
observation = generate_using_text(json_docs, question)
print(f'{observation}')

I0000 00:00:1720806127.507644   14787 ev_epoll1_linux.cc:125] grpc epoll fd: 85


2024-07-12 18:43:07,314 - root - INFO - Response Prompt Token Count: 1736912
2024-07-12 18:43:07,317 - root - INFO - Candidates Token Count: 526
2024-07-12 18:43:07,319 - root - INFO - Total Token Count: 1737438
Based on the safety observations from the Seven Arctic and Normand Subsea, the most pressing issue to address on your worksites is **housekeeping**. This recurring problem presents various safety hazards, including:

* **Trip hazards:** Loose items, cables, hoses, and tools left on walkways and stairwells pose a significant risk of trips and falls. 
    *  *SevenArctic Observation ID 2:  "Found tap in bridge cleaning locker not properly closed and dripping. Waste of water. "*
    *  *SevenArctic Observation ID 10: "Loose grating on deck of messroom to left of hot counter"*
    *  *SevenArctic Observation ID 71:  "Hose left over EEBD box blocking access"*
    *  *Normand Subsea Observation 122: "Found garbage bin in Bosun store showing no clear signs of segregation"*

* **Droppe

I0000 00:00:1720806187.327571   15872 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720806187.331027   15872 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


## Try as YAML

In [19]:

df_yaml = yaml.dump(clean_nones(df1.to_dict(orient='records')),default_flow_style=None)

In [20]:
print(f'{df_yaml[:1000]}')

- {Certification Issue: .nan, Communication: .nan, Conformance to rules: .nan, Date: !!timestamp '2021-01-02
    00:00:00', Defective tools /Equipment: .nan, Department: Project - Deck, Description of observation: Suggestion
    to weld a padeye directly below MHS main lift.  This will eliminate trip hazards
    and give a direct seafastening point., Equipment/Material; issue: .nan, Eyes on path: .nan,
  Eyes on task: .nan, Further Action Required: 'This will be reviewed first by the
    Deck Fmn and OM before any actions are taken. ', Housekeeping: .nan, Immediate Action: Spoke
    with the welder to look into modifying grating to suit padeye., Improvement suggestion: S,
  Inadequate guard's and Barriers: .nan, Line of Fire: .nan, Name of observer: Brian
    Bullock, Objects with potential to fall: .nan, Obs. No: 120.0, Observation close out comment/date input: &id001 !!timestamp '2021-03-04
    00:00:00', PPE Eye / Face Protection: .nan, PPE Feet / Ankle Protection: .nan,
  PPE Hand 

In [21]:
observation = generate_using_text(df_yaml, "what is the most unsafe location")
print(observation)

I0000 00:00:1720806187.964836   14787 ev_epoll1_linux.cc:125] grpc epoll fd: 71


2024-07-12 18:44:53,077 - root - INFO - Response Prompt Token Count: 121743
2024-07-12 18:44:53,078 - root - INFO - Candidates Token Count: 266
2024-07-12 18:44:53,078 - root - INFO - Total Token Count: 122009
It is impossible to determine the most unsafe location from the provided safety observations. The observations highlight various safety concerns in different departments and locations, but there is no clear indication of one location being the most unsafe. 

Here are a few examples of observations from different locations:

* **Project - Deck:** Suggestion to weld a padeye below the MHS main lift to eliminate trip hazards (Obs. No. 120.0)
* **Tooling:**  Concern about the inability to test pull force required to remove equipment after anode skids are deployed (Obs. No. 121.0)
* **Marine - Engine:** Report of warm water leaking in the public toilet on the 1st deck (Obs. No. 123.0)
* **Marine - Bridge:** Two instances of power loss within the Glen Lyon swing circle, highlighting th

I0000 00:00:1720806293.081313   15899 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720806293.102917   15899 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce
